In [ ]:
"""
Module: Loan Risk Inference Pipeline (API Simulation)
Author: Lincoln
Description: Simulates a production inference server loading a frozen PySpark 
             model artifact and executing batch predictions on raw incoming data.
"""

from pyspark.sql import SparkSession
from pyspark.ml import PipelineModel
from pyspark.sql.functions import when, col

# Initialize inference runtime environment
spark = SparkSession.builder \
    .appName("Loan-Risk-Inference-Server") \
    .getOrCreate()

# Load serialized model pipeline from production directory
model_save_path = r"C:\Users\Lincoln\loan prediction dataset\production_loan_model"
production_model = PipelineModel.load(model_save_path)

# Ingest fresh batch inference dataset
new_dataset_path = r"C:\Users\Lincoln\Downloads\manipulated_loan_testing_data.csv"
raw_applicants = spark.read.csv(new_dataset_path, header=True, inferSchema=True)

# Schema alignment step for upstream interface consistency
aligned_applicants = raw_applicants \
    .withColumnRenamed("Credit_Hist", "Credit_History") \
    .withColumnRenamed("Dependent", "Dependents") \
    .withColumnRenamed("Self_Emplo", "Self_Employed")

# Impute missing records using runtime defaults
cleaned_applicants = aligned_applicants.na.fill({
    'LoanAmount': 146, 
    'Loan_Amount_Term': 360, 
    'Credit_History': 1
}).na.fill("Unknown", subset=['Gender', 'Married', 'Dependents', 'Self_Employed'])

# Construct continuous numerical features matching model schema
inference_ready_df = cleaned_applicants.withColumn(
    "DebtToIncomeRatio", 
    (col("LoanAmount") * 1000) / (col("ApplicantIncome") + col("CoapplicantIncome") + 1.0)
)

# Execute model prediction step
predictions = production_model.transform(inference_ready_df)

# Map mathematical classifications back to enterprise application layout
final_decisions = predictions.withColumn(
    "Automated_Bank_Decision", 
    when(col("prediction") == 0.0, "Approved").otherwise("Denied")
)

# Export analytical display schema
final_decisions.select(
    "Loan_ID", 
    "ApplicantIncome", 
    "Credit_History", 
    "LoanAmount", 
    "DebtToIncomeRatio", 
    "Automated_Bank_Decision"
).show(20, False)